In [ ]:
# Project setup
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from src.data.loaders import PointsDataset
from src.visualization.visualization_utils import create_meshgrid
from src.visualization.feature_space_plots import plot_in_rectangular_coordinates, PlotManager
from src.data.synthetic import generate_custom_polar_dataset
from src.features.transformations import cartesian_to_polar, polar_to_curvilinear_r2cos2theta, polar_to_cartesian

from src.visualization.decision_boundaries import plot_decision_boundary
from sklearn.linear_model import LogisticRegression
from src.evaluation.metrics import ModelMetrics
from src.models.model_utilities import Model1DWrapper
from sklearn.svm import SVC

In [ ]:
random_state = 42

current_dir = Path.cwd()

PLOT_CONFIG = {
    "show": True,
    "save": True,
    "out_dir": current_dir.parent / "results" / "figures"
}

plot_manager = PlotManager(PLOT_CONFIG)

# Dataset Sample with Noise

In [ ]:
r_cutoff = 1.0
r_max = 2 * r_cutoff
theta_kappa = 1000
r_noise = 0.1
num_class_points = 80

In [ ]:
synth_polar_pos_dataset = generate_custom_polar_dataset(
    label=1,
    num_points=num_class_points,
    theta_noise=theta_kappa,
    r_noise=r_noise,
    r_outer_bound_func=lambda theta: np.array([r_cutoff]*theta.size),
    random_seed=random_state
)

synth_polar_neg_dataset = generate_custom_polar_dataset(
    label=0,
    num_points=num_class_points,
    r_inner_bound_func=lambda theta: np.array([r_cutoff]*theta.size),
    r_outer_bound_func=lambda theta: np.array([r_max]*theta.size),
    theta_noise=theta_kappa,
    r_noise=r_noise,
    random_seed=random_state
)

synth_polar_dataset = PointsDataset.merge_dataset(synth_polar_pos_dataset, synth_polar_neg_dataset)
synth_cartesian_dataset = synth_polar_dataset.transform(polar_to_cartesian)
synth_r2cos2_dataset = synth_polar_dataset.transform(polar_to_curvilinear_r2cos2theta)

In [ ]:
sample_fig, sample_ax = plot_in_rectangular_coordinates(
    synth_cartesian_dataset.X, 
    synth_cartesian_dataset.y,
    axis_x_label = 'x', axis_y_label = 'y',
    title=""
)

Thetas = np.linspace(-np.pi, np.pi, 100)
sample_ax.plot(r_cutoff*np.cos(Thetas), r_cutoff*np.sin(Thetas), color='black', linestyle='--', label='Generative boundary')
sample_ax.legend()
sample_ax.set_aspect('equal')

display(sample_fig)

In [ ]:
plot_manager.handle(sample_fig, name='sample_synthetic_dataset')

In [ ]:
TP_ref = len(synth_polar_dataset.X[(synth_polar_dataset.X['r']<=r_cutoff) & (synth_polar_dataset.y==1)])
TN_ref = len(synth_polar_dataset.X[(synth_polar_dataset.X['r']>r_cutoff) & (synth_polar_dataset.y==0)])

reference_metrics = {
    'accuracy': (TP_ref + TN_ref)/(2*num_class_points),
    'precision': TP_ref/(TP_ref + (num_class_points-TN_ref)),
    'recall': TP_ref/(num_class_points)
}
reference_metrics['f1'] = 2 * (reference_metrics['precision'] * reference_metrics['recall']) / (reference_metrics['precision'] + reference_metrics['recall'])
metrics_df = pd.DataFrame(reference_metrics, index=['Generator decision boundary'])
display(metrics_df)

## Decision Boundaries

In [ ]:
# Create a Cartesian meshgrid to plot decision boundaries
grid_x, grid_y, grid_cartesian = create_meshgrid(x1=synth_cartesian_dataset.X['x'], x2=synth_cartesian_dataset.X['y'], resolution=500)
grid_cartesian = grid_cartesian.rename(columns={'x1': 'x', 'x2': 'y'})

# Transform meshgrid into polar grid
grid_polar = cartesian_to_polar(grid_cartesian)

# Transform polar into curvilinear grid
grid_r2cos2_sqd = pd.DataFrame({'r^2': grid_polar['r']**2, 'cos^2()': np.cos(grid_polar['theta'])**2})

### Logistic

In [ ]:
lr_model = LogisticRegression(random_state=random_state)
lr_model.fit(synth_polar_dataset.X, synth_polar_dataset.y)
lr_metrics = ModelMetrics('Logistic Regression', 'Polar', synth_polar_dataset.y, lr_model.predict(synth_polar_dataset.X))
lr_metrics.compute_metrics()
metrics_df = pd.concat([metrics_df, lr_metrics.to_pandas().rename(index={0:'Logistic Regression | Polar'})])

lr_fig, lr_ax = plot_decision_boundary(
    model=lr_model, dataset_X=synth_cartesian_dataset.X, dataset_y=synth_cartesian_dataset.y,
    prediction_domain=grid_polar, grid_x=grid_x, grid_y=grid_y,
    title=f'model: Logistic | dataset: Synthetic Circular Polar',
    metrics=lr_metrics.to_dict()
)

In [ ]:
lr_ax.set_aspect('equal')
display(lr_fig)

In [ ]:
plot_manager.handle(lr_fig, name='logi_regr_synthetic_circular_polar_dataset')

### Logistic on Radius Only

In [ ]:
lr_modelR = LogisticRegression(random_state=random_state)
lr_modelR.fit(synth_polar_dataset.X[['r']], synth_polar_dataset.y)
lr_metricsR = ModelMetrics('Logistic Regression', 'R-only', synth_polar_dataset.y, lr_modelR.predict(synth_polar_dataset.X[['r']]))
lr_metricsR.compute_metrics()
metrics_df = pd.concat([metrics_df, lr_metricsR.to_pandas().rename(index={0:'Logistic Regression | R-only'})])

wrapped_modelR = Model1DWrapper(trained_model=lr_modelR, vars_list=['r'])

lr_R_fig, lr_R_ax = plot_decision_boundary(
    model=wrapped_modelR, dataset_X=synth_cartesian_dataset.X, dataset_y=synth_cartesian_dataset.y,
    prediction_domain=grid_polar, grid_x=grid_x, grid_y=grid_y,
    title=f'model: Logistic | dataset: Synthetic Circular R-only',
    metrics=lr_metricsR.to_dict()
)

In [ ]:
lr_R_ax.set_aspect('equal')
display(lr_R_fig)

In [ ]:
plot_manager.handle(lr_R_fig, name='logi_regr_synthetic_circular_r_dataset')

### Logistic Regression on Featured Coordinates

In [ ]:
lr_model_r2cos2 = LogisticRegression(random_state=random_state)
lr_model_r2cos2.fit(synth_r2cos2_dataset.X, synth_r2cos2_dataset.y)
lr_metrics_r2cos2 = ModelMetrics('Logistic Regression', 'r^2,cos2()', synth_r2cos2_dataset.y, lr_model_r2cos2.predict(synth_r2cos2_dataset.X))
lr_metrics_r2cos2.compute_metrics()
metrics_df = pd.concat([metrics_df, lr_metrics_r2cos2.to_pandas().rename(index={0:'Logistic Regression | Curvilinear'})])

dataset_label = r'$\cos^2{\theta}$'

lr_r2cos2_fig, lr_r2cos2_ax = plot_decision_boundary(
    model=lr_model_r2cos2, dataset_X=synth_cartesian_dataset.X, dataset_y=synth_cartesian_dataset.y,
    prediction_domain=grid_r2cos2_sqd, grid_x=grid_x, grid_y=grid_y,
    title=f'model: Logistic | dataset: Synthetic Circular Curvilinear',
    metrics=lr_metrics_r2cos2.to_dict()
)

In [ ]:
lr_r2cos2_ax.set_aspect('equal')
display(lr_r2cos2_fig)

In [ ]:
plot_manager.handle(lr_r2cos2_fig, name='logi_regr_synthetic_circular_curvilinear_dataset')

### SVM

In [ ]:
svm_model = SVC(kernel='rbf', random_state=random_state)
svm_model.fit(synth_cartesian_dataset.X, synth_cartesian_dataset.y)
svm_metrics = ModelMetrics('SVM', 'Cartesian', synth_cartesian_dataset.y, svm_model.predict(synth_cartesian_dataset.X))
svm_metrics.compute_metrics()
metrics_df = pd.concat([metrics_df, svm_metrics.to_pandas().rename(index={0:'SVM | Cartesian'})])

svm_fig, svm_ax = plot_decision_boundary(
    model=svm_model, dataset_X=synth_cartesian_dataset.X, dataset_y=synth_cartesian_dataset.y,
    prediction_domain=grid_cartesian, grid_x=grid_x, grid_y=grid_y,
    title=f'model: SVM | dataset: Synthetic Cartesian',
    metrics=svm_metrics.to_dict()
)

In [ ]:
svm_ax.set_aspect('equal')
display(svm_fig)

In [ ]:
plot_manager.handle(svm_fig, name='svm_synthetic_circular_cartesian_dataset')

### Sample Summary

In [ ]:
display(metrics_df.loc[~metrics_df.index.duplicated(keep='first')])

# Radially Symmetric Data with Noise Model Evaluation

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate

### Small Datasets

In [ ]:
r_cutoff = 1.0
r_max = 2 * r_cutoff
theta_kappa = 1000
r_noise = 0.1
num_class_points = 100

In [ ]:
synth_polar_pos_dataset = generate_custom_polar_dataset(
    label=1,
    num_points=num_class_points,
    theta_noise=theta_kappa,
    r_noise=r_noise,
    r_outer_bound_func=lambda theta: np.array([r_cutoff]*theta.size),
    random_seed=random_state
)

synth_polar_neg_dataset = generate_custom_polar_dataset(
    label=0,
    num_points=num_class_points,
    r_inner_bound_func=lambda theta: np.array([r_cutoff]*theta.size),
    r_outer_bound_func=lambda theta: np.array([r_max]*theta.size),
    theta_noise=theta_kappa,
    r_noise=r_noise,
    random_seed=random_state
)

synth_polar_dataset = PointsDataset.merge_dataset(synth_polar_pos_dataset, synth_polar_neg_dataset)
synth_cartesian_dataset = synth_polar_dataset.transform(polar_to_cartesian)
synth_r2cos2_dataset = synth_polar_dataset.transform(polar_to_curvilinear_r2cos2theta)

In [ ]:
sample_fig, sample_ax = plot_in_rectangular_coordinates(
    synth_cartesian_dataset.X, 
    synth_cartesian_dataset.y,
    title=""
)

Thetas = np.linspace(-np.pi, np.pi, 100)
sample_ax.plot(r_cutoff*np.cos(Thetas), r_cutoff*np.sin(Thetas), color='black', linestyle='--', label='Natural boundary')
sample_ax.legend()
sample_ax.set_aspect('equal')

display(sample_fig)

In [ ]:
plot_manager.handle(sample_fig, name='synthetic_circular_small_dataset')

In [ ]:
# Create a Cartesian meshgrid to plot decision boundaries
grid_x, grid_y, grid_cartesian = create_meshgrid(x1=synth_cartesian_dataset.X['x'], x2=synth_cartesian_dataset.X['y'], resolution=500)
grid_cartesian = grid_cartesian.rename(columns={'x1': 'x', 'x2': 'y'})

# Transform meshgrid into polar grid
grid_polar = cartesian_to_polar(grid_cartesian)

# Transform polar into curvilinear grid
grid_r2cos2_sqd = pd.DataFrame({'r^2': grid_polar['r']**2, 'cos^2()': np.cos(grid_polar['theta'])**2})

#### Cross Validation

In [ ]:
# Initialize with 5 folds
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=random_state
)

folds = list(skf.split(synth_cartesian_dataset.X, synth_cartesian_dataset.y))

In [ ]:
logreg_polar_pipeline = Pipeline([
    ("logreg_polar", LogisticRegression())
])
scoring = ["accuracy", "precision", "recall", "f1"]
logreg_polar_results = cross_validate(
    logreg_polar_pipeline,
    synth_polar_dataset.X,
    synth_polar_dataset.y,
    cv=folds,
    scoring=scoring,
    return_train_score=True
)

logreg_R_pipeline = Pipeline([
    ("logreg_R", LogisticRegression())
])
scoring = ["accuracy", "precision", "recall", "f1"]
logreg_R_results = cross_validate(
    logreg_R_pipeline,
    synth_polar_dataset.X[['r']],
    synth_polar_dataset.y,
    cv=folds,
    scoring=scoring,
    return_train_score=True
)

logreg_r2cos2_pipeline = Pipeline([
    ("logreg_cos2", LogisticRegression())
])
scoring = ["accuracy", "precision", "recall", "f1"]
logreg_r2cos2_results = cross_validate(
    logreg_r2cos2_pipeline,
    synth_r2cos2_dataset.X,
    synth_r2cos2_dataset.y,
    cv=folds,
    scoring=scoring,
    return_train_score=True
)

svm_pipeline = Pipeline([
    ("svm", SVC(kernel="rbf"))
])
scoring = ["accuracy", "precision", "recall", "f1"]
svm_results = cross_validate(
    svm_pipeline,
    synth_cartesian_dataset.X,
    synth_cartesian_dataset.y,
    cv=folds,
    scoring=scoring,
    return_train_score=True
)

In [ ]:
models_test = {}
models_train = {}

for key in svm_results.keys():
    svm_metrics = svm_results[key]
    logreg_metrics = logreg_polar_results[key]
    logreg_R_metrics = logreg_R_results[key]
    logreg_cos2_metrics = logreg_r2cos2_results[key]

    if 'train' in key:
        models_train[key] = {}
        my_dict = models_train
    else:
        models_test[key] = {}
        my_dict = models_test

    my_dict[key] = [f'{np.mean(svm_metrics):.5f} +/- {np.std(svm_metrics):.5f}',
                    f'{np.mean(logreg_metrics):.5f} +/- {np.std(logreg_metrics):.5f}', 
                    f'{np.mean(logreg_R_metrics):.5f} +/- {np.std(logreg_R_metrics):.5f}', 
                    f'{np.mean(logreg_cos2_metrics):.5f} +/- {np.std(logreg_cos2_metrics):.5f}']

model_names = ['SVM', 'LogReg-Polar', 'LogReg-R-only', 'LogReg-r2cos2']

models_test_df = pd.DataFrame(models_test, index=model_names)
models_train_df = pd.DataFrame(models_train, index=model_names)

display(models_test_df)
display(models_train_df)

### Medium Datasets

In [ ]:
num_class_points = 700
r_cutoff = 1.0
r_max = 2 * r_cutoff
theta_kappa = 1000
r_noise = 0.1

In [ ]:
synth_polar_pos_dataset = generate_custom_polar_dataset(
    label=1,
    num_points=num_class_points,
    theta_noise=theta_kappa,
    r_noise=r_noise,
    r_outer_bound_func=lambda theta: np.array([r_cutoff]*theta.size)
)

synth_polar_neg_dataset = generate_custom_polar_dataset(
    label=0,
    num_points=num_class_points,
    r_inner_bound_func=lambda theta: np.array([r_cutoff]*theta.size),
    r_outer_bound_func=lambda theta: np.array([r_max]*theta.size),
    theta_noise=theta_kappa,
    r_noise=r_noise
)

synth_polar_dataset = PointsDataset.merge_dataset(synth_polar_pos_dataset, synth_polar_neg_dataset)
synth_cartesian_dataset = synth_polar_dataset.transform(polar_to_cartesian)
synth_r2cos2_dataset = synth_polar_dataset.transform(polar_to_curvilinear_r2cos2theta)

In [ ]:
sample_fig, sample_ax = plot_in_rectangular_coordinates(
    synth_cartesian_dataset.X, 
    synth_cartesian_dataset.y,
    title=""
)

Thetas = np.linspace(-np.pi, np.pi, 100)
sample_ax.plot(r_cutoff*np.cos(Thetas), r_cutoff*np.sin(Thetas), color='black', linestyle='--', label='Natural boundary')
sample_ax.legend()
sample_ax.set_aspect('equal')

display(sample_fig)

In [ ]:
plot_manager.handle(sample_fig, name='synthetic_circular_medium_dataset')

In [ ]:
# Create a Cartesian meshgrid to plot decision boundaries
grid_x, grid_y, grid_cartesian = create_meshgrid(x1=synth_cartesian_dataset.X['x'], x2=synth_cartesian_dataset.X['y'], resolution=500)
grid_cartesian = grid_cartesian.rename(columns={'x1': 'x', 'x2': 'y'})

# Transform meshgrid into polar grid
grid_polar = cartesian_to_polar(grid_cartesian)

# Transform polar into curvilinear grid
grid_r2cos2_sqd = pd.DataFrame({'r^2': grid_polar['r']**2, 'cos^2()': np.cos(grid_polar['theta'])**2})

#### Cross Validation

In [ ]:
# Initialize with 10 folds
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=random_state
)

folds = list(skf.split(synth_cartesian_dataset.X, synth_cartesian_dataset.y))

In [ ]:
logreg_polar_pipeline = Pipeline([
    ("logreg_polar", LogisticRegression())
])
scoring = ["accuracy", "precision", "recall", "f1"]
logreg_polar_results = cross_validate(
    logreg_polar_pipeline,
    synth_polar_dataset.X,
    synth_polar_dataset.y,
    cv=folds,
    scoring=scoring,
    return_train_score=True
)

logreg_R_pipeline = Pipeline([
    ("logreg_R", LogisticRegression())
])
scoring = ["accuracy", "precision", "recall", "f1"]
logreg_R_results = cross_validate(
    logreg_R_pipeline,
    synth_polar_dataset.X[['r']],
    synth_polar_dataset.y,
    cv=folds,
    scoring=scoring,
    return_train_score=True
)

logreg_cos2_pipeline = Pipeline([
    ("logreg_cos2", LogisticRegression())
])
scoring = ["accuracy", "precision", "recall", "f1"]
logreg__cos2_results = cross_validate(
    logreg_cos2_pipeline,
    synth_r2cos2_dataset.X,
    synth_r2cos2_dataset.y,
    cv=folds,
    scoring=scoring,
    return_train_score=True
)

svm_pipeline = Pipeline([
    ("svm", SVC(kernel="rbf"))
])
scoring = ["accuracy", "precision", "recall", "f1"]
svm_results = cross_validate(
    svm_pipeline,
    synth_cartesian_dataset.X,
    synth_cartesian_dataset.y,
    cv=folds,
    scoring=scoring,
    return_train_score=True
)

In [ ]:
models_test = {}
models_train = {}

for key in svm_results.keys():
    svm_metrics = svm_results[key]
    logreg_metrics = logreg_polar_results[key]
    logreg_R_metrics = logreg_R_results[key]
    logreg_cos2_metrics = logreg__cos2_results[key]

    if 'train' in key:
        models_train[key] = {}
        my_dict = models_train
    else:
        models_test[key] = {}
        my_dict = models_test

    my_dict[key] = [f'{np.mean(svm_metrics):.5f} +/- {np.std(svm_metrics):.5f}',
                    f'{np.mean(logreg_metrics):.5f} +/- {np.std(logreg_metrics):.5f}', 
                    f'{np.mean(logreg_R_metrics):.5f} +/- {np.std(logreg_R_metrics):.5f}', 
                    f'{np.mean(logreg_cos2_metrics):.5f} +/- {np.std(logreg_cos2_metrics):.5f}']

model_names = ['SVM', 'LogReg-Polar', 'LogReg-R-only', 'LogReg-r2cos2']

models_test_df = pd.DataFrame(models_test, index=model_names)
models_train_df = pd.DataFrame(models_train, index=model_names)

display(models_test_df)
display(models_train_df)

### Large Datasets

In [ ]:
num_class_points = 4000
r_cutoff = 1.0
r_max = 2 * r_cutoff
theta_kappa = 1000
r_noise = 0.1

In [ ]:
synth_polar_pos_dataset = generate_custom_polar_dataset(
    label=1,
    num_points=num_class_points,
    theta_noise=theta_kappa,
    r_noise=r_noise,
    r_outer_bound_func=lambda theta: np.array([r_cutoff]*theta.size)
)

synth_polar_neg_dataset = generate_custom_polar_dataset(
    label=0,
    num_points=num_class_points,
    r_inner_bound_func=lambda theta: np.array([r_cutoff]*theta.size),
    r_outer_bound_func=lambda theta: np.array([r_max]*theta.size),
    theta_noise=theta_kappa,
    r_noise=r_noise
)

synth_polar_dataset = PointsDataset.merge_dataset(synth_polar_pos_dataset, synth_polar_neg_dataset)
synth_cartesian_dataset = synth_polar_dataset.transform(polar_to_cartesian)
synth_r2cos2_dataset = synth_polar_dataset.transform(polar_to_curvilinear_r2cos2theta)

In [ ]:
sample_fig, sample_ax = plot_in_rectangular_coordinates(
    synth_cartesian_dataset.X, 
    synth_cartesian_dataset.y,
    title=""
)

Thetas = np.linspace(-np.pi, np.pi, 100)
sample_ax.plot(r_cutoff*np.cos(Thetas), r_cutoff*np.sin(Thetas), color='black', linestyle='--', label='Natural boundary', zorder=5)
sample_ax.legend()
sample_ax.set_aspect('equal')

display(sample_fig)

In [ ]:
plot_manager.handle(sample_fig, name='synthetic_circular_large_dataset')

In [ ]:
# Create a Cartesian meshgrid to plot decision boundaries
grid_x, grid_y, grid_cartesian = create_meshgrid(x1=synth_cartesian_dataset.X['x'], x2=synth_cartesian_dataset.X['y'], resolution=500)
grid_cartesian = grid_cartesian.rename(columns={'x1': 'x', 'x2': 'y'})

# Transform meshgrid into polar grid
grid_polar = cartesian_to_polar(grid_cartesian)

# Transform polar into curvilinear grid
grid_r2cos2_sqd = pd.DataFrame({'r^2': grid_polar['r']**2, 'cos^2()': np.cos(grid_polar['theta'])**2})

#### Cross Validation

In [ ]:
# Initialize with 10 folds
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=random_state
)

folds = list(skf.split(synth_cartesian_dataset.X, synth_cartesian_dataset.y))

In [ ]:
logreg_polar_pipeline = Pipeline([
    ("logreg_polar", LogisticRegression())
])
scoring = ["accuracy", "precision", "recall", "f1"]
logreg_polar_results = cross_validate(
    logreg_polar_pipeline,
    synth_polar_dataset.X,
    synth_polar_dataset.y,
    cv=folds,
    scoring=scoring,
    return_train_score=True
)

logreg_R_pipeline = Pipeline([
    ("logreg_R", LogisticRegression())
])
scoring = ["accuracy", "precision", "recall", "f1"]
logreg_R_results = cross_validate(
    logreg_R_pipeline,
    synth_polar_dataset.X[['r']],
    synth_polar_dataset.y,
    cv=folds,
    scoring=scoring,
    return_train_score=True
)

logreg_cos2_pipeline = Pipeline([
    ("logreg_cos2", LogisticRegression())
])
scoring = ["accuracy", "precision", "recall", "f1"]
logreg__cos2_results = cross_validate(
    logreg_cos2_pipeline,
    synth_r2cos2_dataset.X,
    synth_r2cos2_dataset.y,
    cv=folds,
    scoring=scoring,
    return_train_score=True
)

svm_pipeline = Pipeline([
    ("svm", SVC(kernel="rbf"))
])
scoring = ["accuracy", "precision", "recall", "f1"]
svm_results = cross_validate(
    svm_pipeline,
    synth_cartesian_dataset.X,
    synth_cartesian_dataset.y,
    cv=folds,
    scoring=scoring,
    return_train_score=True
)

In [ ]:
models_test = {}
models_train = {}

for key in svm_results.keys():
    svm_metrics = svm_results[key]
    logreg_metrics = logreg_polar_results[key]
    logreg_R_metrics = logreg_R_results[key]
    logreg_cos2_metrics = logreg__cos2_results[key]

    if 'train' in key:
        models_train[key] = {}
        my_dict = models_train
    else:
        models_test[key] = {}
        my_dict = models_test

    my_dict[key] = [f'{np.mean(svm_metrics):.5f} +/- {np.std(svm_metrics):.5f}',
                    f'{np.mean(logreg_metrics):.5f} +/- {np.std(logreg_metrics):.5f}', 
                    f'{np.mean(logreg_R_metrics):.5f} +/- {np.std(logreg_R_metrics):.5f}', 
                    f'{np.mean(logreg_cos2_metrics):.5f} +/- {np.std(logreg_cos2_metrics):.5f}']

model_names = ['SVM', 'LogReg-Polar', 'LogReg-R-only', 'LogReg-r2cos2']

models_test_df = pd.DataFrame(models_test, index=model_names)
models_train_df = pd.DataFrame(models_train, index=model_names)

display(models_test_df)
display(models_train_df)

# Elliptical Data

In [ ]:
num_class_points = 200
r_cutoff = 1.0
r_max = 2 * r_cutoff
theta_kappa = 3000
r_noise = 0.1

In [ ]:
# Ellptical boundary definition
a = 1.8
b = 0.6
parametric_x_bound_func = lambda theta: a*np.cos(theta)
parametric_y_bound_func = lambda theta: b*np.sin(theta)
gen_rbound_func = lambda theta: (a*b)/(np.sqrt((b*np.cos(theta))**2 + (a*np.sin(theta))**2))

synth_polar_pos_dataset = generate_custom_polar_dataset(
    label=1,
    num_points=num_class_points,
    theta_noise=theta_kappa,
    r_noise=r_noise,
    r_outer_bound_func=gen_rbound_func,
    random_seed=random_state
)

synth_polar_neg_dataset = generate_custom_polar_dataset(
    label=0,
    num_points=num_class_points,
    r_inner_bound_func=gen_rbound_func,
    r_outer_bound_func=lambda theta: np.array([3]*theta.size),
    theta_noise=theta_kappa,
    r_noise=r_noise,
    random_seed=random_state
)

synth_polar_dataset = PointsDataset.merge_dataset(synth_polar_pos_dataset, synth_polar_neg_dataset)
synth_cartesian_dataset = synth_polar_dataset.transform(polar_to_cartesian)
synth_r2cos2_dataset = synth_polar_dataset.transform(polar_to_curvilinear_r2cos2theta)

In [ ]:
elliptic_fig, elliptic_ax = plot_in_rectangular_coordinates(
    synth_cartesian_dataset.X, 
    synth_cartesian_dataset.y,
    title=""
)

Thetas = np.linspace(-np.pi, np.pi, 100)
Rs = (a*b)/(np.sqrt((b*np.cos(Thetas))**2 + (a*np.sin(Thetas))**2))
elliptic_ax.plot(Rs*np.cos(Thetas), Rs*np.sin(Thetas), color='black', linestyle='--', label='Generative boundary')
elliptic_ax.legend()
elliptic_ax.set_aspect('equal')

display(elliptic_fig)

In [ ]:
plot_manager.handle(elliptic_fig, name='sample_synthetic_elliptical_dataset')

In [ ]:
# Create a Cartesian meshgrid to plot decision boundaries
grid_x, grid_y, grid_cartesian = create_meshgrid(x1=synth_cartesian_dataset.X['x'], x2=synth_cartesian_dataset.X['y'], resolution=500)
grid_cartesian = grid_cartesian.rename(columns={'x1': 'x', 'x2': 'y'})

# Transform meshgrid into polar grid
grid_polar = cartesian_to_polar(grid_cartesian)

# Transform polar into curvilinear grid
grid_r2cos2_sqd = pd.DataFrame({'r^2': grid_polar['r']**2, 'cos^2()': np.cos(grid_polar['theta'])**2})

In [ ]:
TP_ref = len(synth_polar_dataset.X[(synth_polar_dataset.X['r']<=gen_rbound_func(synth_polar_dataset.X['theta'])) & (synth_polar_dataset.y==1)])
TN_ref = len(synth_polar_dataset.X[(synth_polar_dataset.X['r']>gen_rbound_func(synth_polar_dataset.X['theta'])) & (synth_polar_dataset.y==0)])

reference_metrics = {
    'accuracy': (TP_ref + TN_ref)/(2*num_class_points),
    'precision': TP_ref/(TP_ref + (num_class_points-TN_ref)),
    'recall': TP_ref/(num_class_points)
}
reference_metrics['f1'] = 2 * (reference_metrics['precision'] * reference_metrics['recall']) / (reference_metrics['precision'] + reference_metrics['recall'])
metrics_df = pd.DataFrame(reference_metrics, index=['Generator decision boundary'])
display(metrics_df)

### Logistic Regression on Polar Data

In [ ]:
lr_model = LogisticRegression(random_state=random_state)
lr_model.fit(synth_polar_dataset.X, synth_polar_dataset.y)
lr_metrics = ModelMetrics('Logistic Regression', 'Polar', synth_polar_dataset.y, lr_model.predict(synth_polar_dataset.X))
lr_metrics.compute_metrics()
metrics_df = pd.concat([metrics_df, lr_metrics.to_pandas().rename(index={0:'Logistic | Polar'})])

lr_fig, lr_ax = plot_decision_boundary(
    model=lr_model, dataset_X=synth_cartesian_dataset.X, dataset_y=synth_cartesian_dataset.y,
    prediction_domain=grid_polar, grid_x=grid_x, grid_y=grid_y,
    title=f'model: Logistic | dataset: Synthetic Elliptical Polar',
    metrics=lr_metrics.to_dict()
)

lr_ax.set_aspect('equal')
display(lr_fig)

In [ ]:
plot_manager.handle(lr_fig, name='logi_regr_polar_synthetic_elliptical_dataset')

### Logistic Regression on Custom Curvilinear Coordinates

In [ ]:
lr_model_r2cos2 = LogisticRegression(random_state=random_state)
lr_model_r2cos2.fit(synth_r2cos2_dataset.X, synth_r2cos2_dataset.y)
lr_metrics_r2cos2 = ModelMetrics('Logistic Regression', 'r^2,cos2()', synth_r2cos2_dataset.y, lr_model_r2cos2.predict(synth_r2cos2_dataset.X))
lr_metrics_r2cos2.compute_metrics()
metrics_df = pd.concat([metrics_df, lr_metrics_r2cos2.to_pandas().rename(index={0:'Logistic | Curvilinear Approximate'})])

dataset_label = r'$\cos^2{\theta}$'

lr_r2cos2_fig, lr_r2cos2_ax = plot_decision_boundary(
    model=lr_model_r2cos2, dataset_X=synth_cartesian_dataset.X, dataset_y=synth_cartesian_dataset.y,
    prediction_domain=grid_r2cos2_sqd, grid_x=grid_x, grid_y=grid_y,
    title=f'model: Logistic | dataset: Synthetic Elliptical $r^2$, {dataset_label}',
    metrics=lr_metrics_r2cos2.to_dict()
)

lr_r2cos2_ax.set_aspect('equal')
display(lr_r2cos2_fig)

In [ ]:
plot_manager.handle(lr_r2cos2_fig, name='logi_regr_curvilinear_synthetic_elliptical_dataset')

### SVM

In [ ]:
svm_model = SVC(kernel='rbf', random_state=random_state)
svm_model.fit(synth_cartesian_dataset.X, synth_cartesian_dataset.y)
svm_metrics = ModelMetrics('SVM', 'Cartesian', synth_cartesian_dataset.y, svm_model.predict(synth_cartesian_dataset.X))
svm_metrics.compute_metrics()
metrics_df = pd.concat([metrics_df, svm_metrics.to_pandas().rename(index={0:'SVM | Cartesian'})])

svm_fig, svm_ax = plot_decision_boundary(
    model=svm_model, dataset_X=synth_cartesian_dataset.X, dataset_y=synth_cartesian_dataset.y,
    prediction_domain=grid_cartesian, grid_x=grid_x, grid_y=grid_y,
    title=f'model: SVM | dataset: Synthetic Elliptical Cartesian',
    metrics=svm_metrics.to_dict()
)

svm_ax.set_aspect('equal')
display(svm_fig)

In [ ]:
plot_manager.handle(svm_fig, name='svm_synthetic_elliptical_dataset')

### General Quadratic Form of Ellipse

In [ ]:
# Transform cartesian into quadratic grid
grid_quad = pd.DataFrame({'x^2': grid_cartesian['x']**2, 'y^2': grid_cartesian['y']**2, 'xy': grid_cartesian['x']*grid_cartesian['y'], 'x': grid_cartesian['x'], 'y': grid_cartesian['y']})

# Transform data points from cartesian into quadratic form
synth_quad_dataset = synth_cartesian_dataset.transform(
    lambda df: pd.DataFrame({'x^2': df['x']**2, 'y^2': df['y']**2, 'xy': df['x']*df['y'], 'x': df['x'], 'y': df['y']})
)

In [ ]:
quad_model = LogisticRegression(penalty=None, random_state=random_state)
quad_model.fit(synth_quad_dataset.X, synth_quad_dataset.y)
quad_metrics = ModelMetrics('Logistic Regression', 'Quadratic', synth_quad_dataset.y, quad_model.predict(synth_quad_dataset.X))
quad_metrics.compute_metrics()
metrics_df = pd.concat([metrics_df, quad_metrics.to_pandas().rename(index={0:'Logistic | Quadratic Cartesian'})])

quad_fig, quad_ax = plot_decision_boundary(
    model=quad_model, dataset_X=synth_cartesian_dataset.X, dataset_y=synth_cartesian_dataset.y,
    prediction_domain=grid_quad, grid_x=grid_x, grid_y=grid_y,
    title=f'model: Logistic | dataset: Synthetic Elliptical Quadratic',
    metrics=quad_metrics.to_dict()
)

quad_ax.set_aspect('equal')
display(quad_fig)

In [ ]:
plot_manager.handle(quad_fig, name='logi_regr_quad_synthetic_elliptical_dataset')

### Summary

In [ ]:
display(metrics_df.loc[~metrics_df.index.duplicated(keep='last')])

# Non-radially Symmetrical Data

In [ ]:
num_sub_class_points = 50

synth_non_rad_polar_neg_dataset1 = generate_custom_polar_dataset(
    label=0,
    num_points=num_sub_class_points,
    theta_lb=-np.pi/2,
    theta_ub=-np.pi/4,
    random_seed=random_state
)
synth_non_rad_polar_neg_dataset2 = generate_custom_polar_dataset(
    label=0,
    num_points=num_sub_class_points,
    theta_lb=np.pi/2,
    theta_ub=3*np.pi/4,
    random_seed=random_state
)
synth_non_rad_polar_neg_dataset = PointsDataset.merge_dataset(synth_non_rad_polar_neg_dataset1, synth_non_rad_polar_neg_dataset2)

synth_non_rad_polar_pos_dataset1 = generate_custom_polar_dataset(
    label=1,
    num_points=num_sub_class_points,
    theta_lb=-np.pi,
    theta_ub=-np.pi/2,
    r_inner_bound_func=lambda theta: 0.2 * np.cos(theta)**2,
    r_outer_bound_func=lambda theta: 0.6 + 0.2 * np.cos(theta)**2,
    random_seed=random_state
)
synth_non_rad_polar_pos_dataset2 = generate_custom_polar_dataset(
    label=1,
    num_points=num_sub_class_points,
    theta_lb=-np.pi/4,
    theta_ub=np.pi/2,
    r_inner_bound_func=lambda theta: 0.2 * np.cos(theta)**2,
    r_outer_bound_func=lambda theta: 0.6 + 0.2 * np.cos(theta)**2,
    random_seed=random_state
)
synth_non_rad_polar_pos_dataset = PointsDataset.merge_dataset(synth_non_rad_polar_pos_dataset1, synth_non_rad_polar_pos_dataset2)

synth_non_rad_polar_dataset = PointsDataset.merge_dataset(synth_non_rad_polar_neg_dataset, synth_non_rad_polar_pos_dataset)
synth_non_rad_cartesian_dataset = synth_non_rad_polar_dataset.transform(polar_to_cartesian)
synth_non_rad_r2cos2_dataset = synth_non_rad_polar_dataset.transform(polar_to_curvilinear_r2cos2theta)

In [ ]:
non_rad_fig, non_rad_ax = plot_in_rectangular_coordinates(
    synth_non_rad_cartesian_dataset.X, 
    synth_non_rad_cartesian_dataset.y,
    title=""
)

non_rad_ax.plot(np.zeros(100), np.linspace(-1, 1, 100), color='black', linestyle='--', label='Generative boundary')
non_rad_ax.plot(np.linspace(-0.7, 0.7, 100), -np.linspace(-0.7, 0.7, 100), color='black', linestyle='--')
non_rad_ax.legend()

display(non_rad_fig)

In [ ]:
plot_manager.handle(non_rad_fig, name='synthetic_non_rad_dataset')

In [ ]:
TP_ref = len(
    synth_non_rad_polar_dataset.X[(synth_non_rad_polar_dataset.X['theta']<=np.pi/2) & (synth_non_rad_polar_dataset.X['theta']>=-np.pi/4) & (synth_non_rad_polar_dataset.y==1)]
) + len(
    synth_non_rad_polar_dataset.X[(synth_non_rad_polar_dataset.X['theta']<=-np.pi/2) & (synth_non_rad_polar_dataset.X['theta']>=-np.pi) & (synth_non_rad_polar_dataset.y==1)]
)
TN_ref = len(
    synth_non_rad_polar_dataset.X[(synth_non_rad_polar_dataset.X['theta']>np.pi/2) & (synth_non_rad_polar_dataset.X['theta']<3*np.pi/4) & (synth_non_rad_polar_dataset.y==0)]
) + len(
    synth_non_rad_polar_dataset.X[(synth_non_rad_polar_dataset.X['theta']>-np.pi/2) & (synth_non_rad_polar_dataset.X['theta']<-np.pi/4) & (synth_non_rad_polar_dataset.y==0)]
)

reference_metrics = {
    'accuracy': (TP_ref + TN_ref)/(4*num_sub_class_points),
    'precision': TP_ref/(TP_ref + (2*num_sub_class_points-TN_ref)),
    'recall': TP_ref/(2*num_sub_class_points)
}
reference_metrics['f1'] = 2 * (reference_metrics['precision'] * reference_metrics['recall']) / (reference_metrics['precision'] + reference_metrics['recall'])
metrics_df = pd.DataFrame(reference_metrics, index=['Generator decision boundary'])
display(metrics_df)

In [ ]:
# Create a Cartesian meshgrid to plot decision boundaries
grid_x, grid_y, grid_cartesian = create_meshgrid(x1=synth_non_rad_cartesian_dataset.X['x'], x2=synth_non_rad_cartesian_dataset.X['y'], resolution=500)
grid_cartesian = grid_cartesian.rename(columns={'x1': 'x', 'x2': 'y'})

# Transform meshgrid into polar grid
grid_polar = cartesian_to_polar(grid_cartesian)

# Transform polar into curvilinear grid
grid_r2cos2_sqd = pd.DataFrame({'r^2': grid_polar['r']**2, 'cos^2()': np.cos(grid_polar['theta'])**2})

In [ ]:
lr_model = LogisticRegression(random_state=random_state)
lr_model.fit(synth_non_rad_polar_dataset.X, synth_non_rad_polar_dataset.y)
lr_metrics = ModelMetrics('Logistic Regression', 'Non-radially Symmetric Polar', synth_non_rad_polar_dataset.y, lr_model.predict(synth_non_rad_polar_dataset.X))
lr_metrics.compute_metrics()
metrics_df = pd.concat([metrics_df, lr_metrics.to_pandas().rename(index={0:'Logistic | Polar'})])

lr_fig, lr_ax = plot_decision_boundary(
    model=lr_model, dataset_X=synth_non_rad_cartesian_dataset.X, dataset_y=synth_non_rad_cartesian_dataset.y,
    prediction_domain=grid_polar, grid_x=grid_x, grid_y=grid_y,
    title=f'model: Logistic | dataset: Non-radially Symmetric Polar',
    metrics=lr_metrics.to_dict()
)

lr_ax.set_aspect('equal')
display(lr_fig)

In [ ]:
plot_manager.handle(lr_fig, name='logi_regr_polar_synthetic_non_rad_dataset')

In [ ]:
lr_model_r2cos2 = LogisticRegression(random_state=random_state)
lr_model_r2cos2.fit(synth_non_rad_r2cos2_dataset.X, synth_non_rad_r2cos2_dataset.y)
lr_metrics_r2cos2 = ModelMetrics('Logistic Regression', 'r^2,cos2()', synth_non_rad_r2cos2_dataset.y, lr_model_r2cos2.predict(synth_non_rad_r2cos2_dataset.X))
lr_metrics_r2cos2.compute_metrics()
metrics_df = pd.concat([metrics_df, lr_metrics_r2cos2.to_pandas().rename(index={0:'Logistic | Curvilinear'})])

dataset_label = r'$\cos^2{\theta}$'

lr_r2cos2_fig, lr_r2cos2_ax = plot_decision_boundary(
    model=lr_model_r2cos2, dataset_X=synth_non_rad_cartesian_dataset.X, dataset_y=synth_non_rad_cartesian_dataset.y,
    prediction_domain=grid_r2cos2_sqd, grid_x=grid_x, grid_y=grid_y,
    title=f'model: Logistic | dataset: Non-radially Symmetric $r^2$, {dataset_label}',
    metrics=lr_metrics_r2cos2.to_dict()
)

lr_r2cos2_ax.set_aspect('equal')
display(lr_r2cos2_fig)

In [ ]:
plot_manager.handle(lr_r2cos2_fig, name='logi_regr_curvilinear_synthetic_non_rad_dataset')

In [ ]:
svm_model = SVC(kernel='rbf', random_state=random_state)
svm_model.fit(synth_non_rad_cartesian_dataset.X, synth_non_rad_cartesian_dataset.y)
svm_metrics = ModelMetrics('SVM', 'Cartesian', synth_non_rad_cartesian_dataset.y, svm_model.predict(synth_non_rad_cartesian_dataset.X))
svm_metrics.compute_metrics()
metrics_df = pd.concat([metrics_df, svm_metrics.to_pandas().rename(index={0:'SVM | Cartesian'})])

svm_fig, svm_ax = plot_decision_boundary(
    model=svm_model, dataset_X=synth_non_rad_cartesian_dataset.X, dataset_y=synth_non_rad_cartesian_dataset.y,
    prediction_domain=grid_cartesian, grid_x=grid_x, grid_y=grid_y,
    title=f'model: SVM | dataset: Non-radially Symmetric Cartesian',
    metrics=svm_metrics.to_dict()
)

svm_ax.set_aspect('equal')
display(svm_fig)

In [ ]:
plot_manager.handle(svm_fig, name='svm_synthetic_non_rad_dataset')

In [ ]:
svm_polar_model = SVC(kernel='rbf', random_state=random_state)
svm_polar_model.fit(synth_non_rad_polar_dataset.X, synth_non_rad_polar_dataset.y)
svm_polar_metrics = ModelMetrics('SVM', 'Polar', synth_non_rad_polar_dataset.y, svm_polar_model.predict(synth_non_rad_polar_dataset.X))
svm_polar_metrics.compute_metrics()
metrics_df = pd.concat([metrics_df, svm_polar_metrics.to_pandas().rename(index={0:'SVM | Polar'})])

svm_polar_fig, svm_polar_ax = plot_decision_boundary(
    model=svm_polar_model, dataset_X=synth_non_rad_cartesian_dataset.X, dataset_y=synth_non_rad_cartesian_dataset.y,
    prediction_domain=grid_polar, grid_x=grid_x, grid_y=grid_y,
    title=f'model: SVM | dataset: Non-radially Symmetric Polar',
    metrics=svm_polar_metrics.to_dict()
)

svm_polar_ax.set_aspect('equal')
display(svm_polar_fig)

In [ ]:
plot_manager.handle(svm_polar_fig, name='svm_synthetic_non_rad_polar_dataset')

In [ ]:
# Transform cartesian into quadratic grid
grid_quad = pd.DataFrame({'x^2': grid_cartesian['x']**2, 'y^2': grid_cartesian['y']**2, 'xy': grid_cartesian['x']*grid_cartesian['y'], 'x': grid_cartesian['x'], 'y': grid_cartesian['y']})

# Transform data points from cartesian into quadratic form
synth_non_rad_quad_dataset = synth_non_rad_cartesian_dataset.transform(
    lambda df: pd.DataFrame({'x^2': df['x']**2, 'y^2': df['y']**2, 'xy': df['x']*df['y'], 'x': df['x'], 'y': df['y']})
)

non_rad_quad_model = LogisticRegression(penalty=None, random_state=random_state)
non_rad_quad_model.fit(synth_non_rad_quad_dataset.X, synth_non_rad_quad_dataset.y)
non_rad_quad_metrics = ModelMetrics('Logistic Regression', 'Quadratic', synth_non_rad_quad_dataset.y, non_rad_quad_model.predict(synth_non_rad_quad_dataset.X))
non_rad_quad_metrics.compute_metrics()
metrics_df = pd.concat([metrics_df, non_rad_quad_metrics.to_pandas().rename(index={0:'Logistic | Quadratic Cartesian'})])

non_rad_quad_fig, non_rad_quad_ax = plot_decision_boundary(
    model=non_rad_quad_model, dataset_X=synth_non_rad_cartesian_dataset.X, dataset_y=synth_non_rad_cartesian_dataset.y,
    prediction_domain=grid_quad, grid_x=grid_x, grid_y=grid_y,
    title=f'model: Logistic | dataset: Non-radially Symmetric Quadratic',
    metrics=non_rad_quad_metrics.to_dict()
)

non_rad_quad_ax.set_aspect('equal')
display(non_rad_quad_fig)

In [ ]:
plot_manager.handle(non_rad_quad_fig, name='logi_regr_synthetic_non_rad_quad_dataset')

## Summary

In [ ]:
display(metrics_df.loc[~metrics_df.index.duplicated(keep='last')])